# SmartHome-Bench 반복 행동 8개 영상 비교

[SmartHome-Bench 데이터셋](https://huggingface.co/datasets/violetcliff/SmartHome-Bench)의 공개 영상 8개를 로컬에서 비교했습니다. 원본 영상은 `data/videos/`에 있으며 Git에는 포함하지 않습니다. API 호출 결과는 `notebooks/smarthome-bench-evidence/`에 저장해 이 노트북을 다시 실행할 때 비용이 들지 않습니다.

**질문:** 8회를 자동으로 정확히 셀 수 있는가? 점프만 어려운가? 다른 행동에서는 언제 되는가?


## 검토 기준과 요약

| 영상 | 행동 | 사람이 확인한 횟수 | 일반 추출기 | 프레임 상태 방식 |
|---|---|---:|---|---:|
| `0936` | 유리문 앞 앞발 들기 | **8** (확실한 점프 6 + 앞발만 2) | 점프 명시 0건, 직접 횟수 질문은 19회 | 9후보, 13초 오탐 |
| `0608` | 수영장 입수 | **2** (약 1.5초·20.5초) | 입수 명시 1건 | 3후보, 11.5~13초 오탐 |
| `0065` | 소파에 올라가기 | **5** (애매한 1회 제외하면 **4**) | 올라가기 명시 1건 | 직접 전환 질문 7후보(검정 6/회색 1) |
| `0142` | 먹던 고양이가 고개 들기 | **2** (약 11~18초·25~28초) | 먹기만 기록, 고개 들기 0건 | **2후보** |
| `0949` | 목줄 당기기 | 미확정 | 7건 중 당기기 0건 | 미실시 |
| `0747` | 고양이의 문초인종 앞발 동작 | 미확정 | 처음 48초에서 앞발 동작 0건 | 미실시 |
| `0133` | 짖기 | 미확정 | 4건 중 짖기 0건 | 프레임만으로 판정 불가 |
| `0037` | 문/카메라 쪽 앞발 동작 | 미확정 | 접근/관찰만 표현 | 시각 오류 및 자세 유지로 횟수 비교 보류 |

`0936`의 8회, `0065`의 4~5회, `0608`·`0142`의 2회는 영상 검토 기준입니다. 모델 결과와 구분합니다. 미확정 영상에는 숫자를 붙이지 않았습니다. `0133`은 데이터셋 설명에 짖는 횟수가 있지만 오디오를 직접 확인하지 않아 정답으로 사용하지 않았습니다.


In [ ]:
import json
from pathlib import Path
from IPython.display import Video, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
evidence = root / "notebooks" / "smarthome-bench-evidence"
review = json.loads((evidence / "review.json").read_text())
for item in review["videos"]:
    print(item["video_id"], item["action"], "사람 확인:", item["human_count"], "|", item["human_source"])
    video = root / item["local_video"]
    if video.exists():
        display(Video(filename=str(video), embed=False, html_attributes="controls"))
    else:
        print("로컬 영상 없음:", video, "| 원본:", item["source_url"])


## 프레임 상태 → 연속 구간

각 프레임을 `active / inactive / unclear`로 판단한 뒤 연속된 `active`를 한 후보로 묶었습니다. 이 단계는 모델이 잘못 판정한 프레임을 바로잡지 못합니다. 아래 코드는 모델 출력의 창 밖 시각과 중복 시각을 제외하고 후보를 계산합니다.


In [ ]:
def state_episodes(path):
    data = json.loads(path.read_text())
    states = {}
    step = data["step"]
    for result in data["results"]:
        lo, hi = result["window"]
        for row in result["states"]:
            t = float(row["t"])
            if lo - 0.001 <= t <= hi + 0.001 and row["state"] in {"active", "inactive", "unclear"}:
                states[t] = row["state"]
    episodes, start, last = [], None, None
    for t, state in sorted(states.items()):
        if state == "active":
            if start is None or (last is not None and t - last > step * 1.5):
                if start is not None:
                    episodes.append((start, last))
                start = t
            last = t
        elif start is not None:
            episodes.append((start, last))
            start = last = None
    if start is not None:
        episodes.append((start, last))
    return episodes

for video_id in ("0936", "0608", "0142", "0037"):
    episodes = state_episodes(evidence / f"smartbench_{video_id}-states.json")
    print(video_id, len(episodes), "후보:", episodes)


## 숫자만 맞아도 실패할 수 있음

`0936`의 9후보를 다른 모델 호출로 재확인하자 8개를 수락했습니다. 하지만 실제 동작이 없는 13초 후보를 수락하고, 실제 19초 동작을 거절했습니다. 아래 셀은 사용자 검토 구간과 후보 위치를 비교합니다. 모델이 8이라고 답한 것을 정확한 추출 성공으로 보지 않는 이유입니다.


In [ ]:
labels = json.loads((evidence / "smartbench_0936-review.json").read_text())["intervals"]
verified = json.loads((evidence / "smartbench_0936-verified.json").read_text())
print("사용자가 검토한 앞발 동작 횟수:", len(labels), "회")
def matches_label(candidate):
    a, b = candidate
    return any(a <= row["end_time"] and b >= row["start_time"] for row in labels)
for row in verified["results"]:
    candidate = row["candidate"]
    accepted = row["assessment"].get("valid") is True
    print(f"{candidate[0]:5.2f}–{candidate[1]:5.2f}", "모델 수락" if accepted else "모델 거절", "| 사용자 구간과 겹침" if matches_label(candidate) else "| 사용자 구간 없음")


## 소파에 올라가기: `smartbench_0065`

사용자 검토 기준은 **검정 소파 3회(1회 애매), 회색 소파 2회**입니다. 애매한 동작을 포함하면 5회, 제외하면 4회입니다. 현재 일반 추출기는 올라가기 이벤트를 1건만 명시했습니다. 0.25초 프레임으로 `바닥→소파` 전환만 묻는 별도 VLM은 겹치는 시각을 합쳐 7후보(검정 6, 회색 1)를 냈습니다. 따라서 숫자와 소파별 분포 모두 사용자 검토와 다릅니다.

아래 후보 중 4.75초와 7초는 이미 소파 위에 있는 장면을 새 올라가기처럼 셌을 가능성이 큽니다. 8.75초는 빠른 출입이 겹쳐 있어 이 노트북에서 오탐으로 단정하지 않습니다. 사용자에게 개별 정답 시각은 받지 않았으므로 정밀한 시각별 정오 평가는 보류합니다.


In [ ]:
sofa = json.loads((evidence / "smartbench_0065-transitions.json").read_text())
raw_events = [event for item in sofa["results"] for event in item["response"].get("events", [])]
unique = {}
for event in raw_events:
    unique[round(float(event["t"]), 2)] = event
print("모델 원시 행:", len(raw_events), "| 중복 시각 합친 후보:", len(unique))
for t, event in sorted(unique.items()):
    print(f"{t:5.2f}s", event["sofa"], "애매" if event["ambiguous"] else "확정 주장")


## 결론과 다음 구현 단계

- **`0936`의 답 8회는 사용자 검토 구간을 세면 얻을 수 있습니다.** 직접 횟수 질문의 모델 답은 19회이고, 상태→구간 방식은 9후보입니다. 검증 모델의 8회는 위치가 틀린 우연한 일치입니다.
- **소파에 올라가기도 틀렸습니다.** `0065`에서 사용자 검토는 4~5회인데 일반 추출기는 1건, 전환 전용 VLM은 7후보였습니다.
- **점프만의 문제는 아닙니다.** `0608` 입수도 2회 대신 3후보를 냈고, `0949` 목줄 당기기 및 `0747` 앞발 동작은 일반 추출기가 명시하지 못했습니다.
- **개선 가능성도 있습니다.** `0142` 고양이 고개 들기는 상태→구간 방식이 사람이 확인한 2구간을 잡았습니다. 다만 이 한 영상만으로 일반적인 정확도를 주장할 수는 없습니다.
- `0133` 짖기는 오디오 정보가 필요합니다. 현재 프레임 전용 추출기로는 정확한 짖음 횟수를 계산할 수 없습니다.
- 개선 알고리즘은 행동마다 `시작 전 상태 → 동작 중 상태 → 복귀 상태`를 정의하고, 짧은 간격으로 프레임을 읽어 후보를 만든 뒤, 각 후보의 시작·끝 프레임을 검증하고 근거 시각과 함께 횟수를 계산하는 것입니다. 여러 검토 영상으로 행동별 정확도를 확인해야 합니다.

**실행 범위:** 기존 DB, worker, 검색 API는 변경하지 않았습니다. 영상 원본은 로컬 파일이며 원본 링크는 `review.json`에 있습니다.
